# Whisper systems: A4, A5 (zero-shot) and C1 (whisper-small fine-tune)

Runs in Colab (A100 for training; RTF for the tables is measured on the GPU named in each run log).
The notebook clones only the folders it needs and reads `data/clean/`; nothing is uploaded by hand.
Checkpoints go to Google Drive so a dropped session can resume.

Rules: settings are chosen on **dev** only. **Test** is run once per system, after the freeze (Friday 20 Nov evening).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/nlp3_whisper'  # checkpoints and copies of benchmark/ outputs
!mkdir -p $OUT

In [ ]:
REPO = 'https://github.com/ata-sesli/nlp-project-3'
BRANCH = 'main'
!git clone --depth 1 --filter=blob:none --sparse -b $BRANCH $REPO /content/nlp-project-3
%cd /content/nlp-project-3
!git sparse-checkout set data/clean data/raw/usc/dev baselines/whisper_zero_shot whisper_finetune benchmark
!ls data data/clean whisper_finetune

In [ ]:
!pip install -q -r whisper_finetune/requirements-whisper.txt
import torch, transformers
print(torch.__version__, transformers.__version__)

In [ ]:
# Hugging Face token from Colab Secrets (key icon on the left, name: HF_TOKEN). Never paste it into a cell.
import os
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

## A4 / A5: zero-shot on dev

In [ ]:
# quick try on 200 clips first
!python baselines/whisper_zero_shot/transcribe.py --model openai/whisper-small --system A4_whisper-small --split dev --limit 200

In [ ]:
!python baselines/whisper_zero_shot/transcribe.py --model openai/whisper-small --system A4_whisper-small --split dev --batch-size 64
!python baselines/whisper_zero_shot/transcribe.py --model openai/whisper-large-v3 --system A5_whisper-large-v3 --split dev --batch-size 32
!head -5 benchmark/predictions/A4_whisper-small__dev.jsonl benchmark/predictions/A5_whisper-large-v3__dev.jsonl
!cp -r benchmark $OUT/

## C1: whisper-small fine-tune

Real training starts only after **Gate 1** (Thu 19 Nov 18:00: `data/clean/train.csv`, `data/normalize.py`, `benchmark/evaluate.py` merged).
Before that, run only the smoke test. Pull the latest branch first so the clean CSVs are used.

In [ ]:
!git pull --ff-only
!git sparse-checkout add data/raw/usc/train
!ls data/clean

In [ ]:
!python whisper_finetune/train.py --output-dir /content/c1_smoke --smoke

In [ ]:
# Full run (after Gate 1). If the session drops, rerun this cell with --resume added.
!python whisper_finetune/train.py --output-dir $OUT/c1 2>&1 | tee -a $OUT/c1_train.log
!cp -r benchmark $OUT/

In [ ]:
HF_REPO = 'CHANGE-ME/whisper-small-uzbek-usc25h'  # your Hugging Face user name
!python whisper_finetune/push_to_hub.py --model-dir $OUT/c1/best --repo-id $HF_REPO
# dev predictions from the uploaded model, so the run log records its revision
!python baselines/whisper_zero_shot/transcribe.py --model $HF_REPO --system C1_whisper-small-ft --split dev --batch-size 64
!cp -r benchmark $OUT/

## Test: once, after the freeze (Sat 21 Nov)

In [ ]:
!git sparse-checkout add data/raw/usc/test
!python baselines/whisper_zero_shot/transcribe.py --model openai/whisper-small --system A4_whisper-small --split test --batch-size 64 --after-freeze
!python baselines/whisper_zero_shot/transcribe.py --model openai/whisper-large-v3 --system A5_whisper-large-v3 --split test --batch-size 32 --after-freeze
!python baselines/whisper_zero_shot/transcribe.py --model $HF_REPO --system C1_whisper-small-ft --split test --batch-size 64 --after-freeze
!python whisper_finetune/push_to_hub.py --model-dir $OUT/c1/best --repo-id $HF_REPO  # card with test scores
!cp -r benchmark $OUT/